# Trips Checks

Checks route-level trip metadata consistency, especially headsign and direction mapping.

In [1]:
from pathlib import Path
import sys

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "checks" / "commons.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/checks/commons.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from data_validation.gtfs_utils import (
    ROUTES_FILE,
    TRIPS_FILE,
    check_missing_files,
    print_file_disclaimer,
    load_route_ids,
    read_dict_rows,
)

In [2]:
from collections import defaultdict
from typing import Dict, List, Set

# Goal: validate that each route_id has a clear and consistent pairing between 
# trip_headsign and direction_id in trips_cleaned.txt.

# Method: for each route_id, we collect all observed trip_headsign values and their 
# direction_id values. We expect two headsigns and two directions, with each headsign 
# mapping to exactly one direction, and both headsigns mapped to different directions.

# Output: for each route, report whether the pairing is correct or show the inconsistencies found.
def main() -> None:
    """Validate trip_headsign and direction_id pairing consistency per route."""
    route_short_name: Dict[str, str] = {}
    by_route: Dict[str, Dict[str, Set[str]]] = {}
    total_trip_rows = 0
    considered_trip_rows = 0
    routes_without_trips = None
    ok_routes = 0
    problematic_routes = 0
    check_missing_files([ROUTES_FILE, TRIPS_FILE])

    print_file_disclaimer([
        (ROUTES_FILE, 'routes'),
        (TRIPS_FILE, 'trips'),
    ])
    
    print(f"\n----- direction_id and trip_headsign from 'trips' are paired? -----")

    for r in read_dict_rows(ROUTES_FILE):

        rid = r.get('route_id', '').strip()
        route_short_name[rid] = r.get('route_short_name', '').strip()

    if not route_short_name:
        print("No route_id found in 'routes'.")
        return

    by_route = defaultdict(lambda: defaultdict(set))
    for r in read_dict_rows(TRIPS_FILE):
        total_trip_rows += 1
        rid = r.get('route_id', '').strip()
        if rid not in route_short_name:
            continue
        considered_trip_rows += 1
        headsign = r.get('trip_headsign', '').strip()
        direction = r.get('direction_id', '').strip()
        by_route[rid][headsign].add(direction)

    print(f"Number of routes in scope: {len(route_short_name)}")
    print(f"Total rows in 'trips': {total_trip_rows}")
    print(f"Rows considered (route_id in scope): {considered_trip_rows}")

    routes_without_trips = sorted(rid for rid in route_short_name if rid not in by_route)
    if routes_without_trips:
        print(f"WARNING: {len(routes_without_trips)} routes in scope have no trips in 'trips'.")
        for rid in routes_without_trips:
            print(f"- {rid} ({route_short_name.get(rid, '')})")

    for rid in sorted(by_route):
        short_name = route_short_name.get(rid, '')
        hd_map = by_route[rid]
        headsigns = sorted(hd_map.keys())
        all_dirs = sorted({d for dirs in hd_map.values() for d in dirs})

        print(f"\nRoute {rid} ({short_name}):")
        print(f"- trip_headsign values: {len(headsigns)}")
        print(f"- direction_id values: {len(all_dirs)}")

        issues: List[str] = []
        if len(headsigns) != 2:
            issues.append(f"expected 2 trip_headsign values, found {len(headsigns)}")
        if len(all_dirs) != 2:
            issues.append(f"expected 2 direction_id values, found {len(all_dirs)}")

        mapping: Dict[str, str] = {}
        ambiguous = []
        for h in headsigns:
            dirs = sorted(d for d in hd_map[h] if d != '')
            if len(dirs) == 1:
                mapping[h] = dirs[0]
            else:
                ambiguous.append((h, dirs))

        for h, dirs in ambiguous:
            issues.append(f"headsign {h!r} maps to multiple direction_id values: {dirs}")

        if len(mapping) == 2 and len(set(mapping.values())) != 2:
            issues.append("both trip_headsign values map to the same direction_id")

        if not issues and len(mapping) == 2:
            h1, h2 = sorted(mapping.keys())
            print("All correct:")
            print(f"- {h1!r} -> direction_id={mapping[h1]!r}")
            print(f"- {h2!r} -> direction_id={mapping[h2]!r}")
            ok_routes += 1
        else:
            problematic_routes += 1
            print(f"MISSING clear headsign-direction pairing for route {rid}:")
            for msg in issues:
                print(f"- {msg}")
            for h in sorted(hd_map.keys()):
                print(f"- {h!r} appears with direction_id values: {sorted(hd_map[h])}")

    print("\nSummary:")
    print(f"- Routes checked with trips: {len(by_route)}")
    print(f"- Routes with clear pairing: {ok_routes}")
    print(f"- Routes with issues: {problematic_routes}")

main()

Disclaimer: for coherence we will consider the next files from /Users/saradalmauguamis/Desktop/Mates/Cursos/Curs 2025-2026 (3r + 4t)/TFG/TFG/.src/gtfs/data:
 - routes_subway.txt as 'routes' from 1_subway
 - trips_cleaned.txt as 'trips' from 2_duplicated_trips

----- direction_id and trip_headsign from 'trips' are paired? -----
Number of routes in scope: 11
Total rows in 'trips': 10977
Rows considered (route_id in scope): 10977

Route 1.1.1 (L1):
- trip_headsign values: 2
- direction_id values: 2
All correct:
- 'Fondo' -> direction_id='0'
- 'Hospital de Bellvitge' -> direction_id='1'

Route 1.101.1 (L10S):
- trip_headsign values: 2
- direction_id values: 2
All correct:
- 'Collblanc' -> direction_id='0'
- 'ZAL | Riu Vell' -> direction_id='1'

Route 1.104.1 (L10N):
- trip_headsign values: 2
- direction_id values: 2
All correct:
- 'Gorg' -> direction_id='0'
- 'La Sagrera' -> direction_id='1'

Route 1.11.1 (L11):
- trip_headsign values: 2
- direction_id values: 2
All correct:
- 'Can Cuiàs' 